In [1]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License.


In [2]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore


In [ ]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import BytesIO

# Azure Blob Storage credentials
# Azure Blob Storage credentials
connection_string = ""
container_name = "output"

# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

def read_parquet_from_azure(file_path):
    """Reads a Parquet file from Azure Blob Storage and returns a Pandas DataFrame."""
    try:
        blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_path)
        parquet_data = blob_client.download_blob().readall()
        df = pd.read_parquet(BytesIO(parquet_data))
        print(f"✅ Successfully read: {file_path}")
        return df
    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return None  # Return None if the file is not found


In [4]:
# File paths in Blob Storage
FINAL_NODE_PATH = "output/create_final_nodes.parquet"
COMMUNITY_REPORT_TABLE = "output/create_final_community_reports.parquet"
COMMUNITY_TABLE = "output/create_final_communities.parquet"
ENTITY_TABLE = "output/create_final_entities.parquet"
RELATIONSHIP_TABLE = "output/create_final_relationships.parquet"
COVARIATE_TABLE = "output/create_final_covariates.parquet"
TEXT_UNIT_TABLE =  "output/create_final_text_units.parquet"
COMMUNITY_LEVEL = 2


In [5]:
# read nodes table to get community and degree data
entity_df = read_parquet_from_azure(ENTITY_TABLE)
community_df = read_parquet_from_azure(COMMUNITY_TABLE)


✅ Successfully read: output/create_final_entities.parquet
✅ Successfully read: output/create_final_communities.parquet


In [6]:
relationship_df = read_parquet_from_azure(RELATIONSHIP_TABLE)
relationships = read_indexer_relationships(relationship_df)


✅ Successfully read: output/create_final_relationships.parquet


In [7]:
%pip install yfiles_jupyter_graphs --quiet
from yfiles_jupyter_graphs import GraphWidget


# converts the entities dataframe to a list of dicts for yfiles-jupyter-graphs
def convert_entities_to_dicts(df):
    """Convert the entities dataframe to a list of dicts for yfiles-jupyter-graphs."""
    nodes_dict = {}
    for _, row in df.iterrows():
        # Create a dictionary for each row and collect unique nodes
        node_id = row["title"]
        if node_id not in nodes_dict:
            nodes_dict[node_id] = {
                "id": node_id,
                "properties": row.to_dict(),
            }
    return list(nodes_dict.values())


# converts the relationships dataframe to a list of dicts for yfiles-jupyter-graphs
def convert_relationships_to_dicts(df):
    """Convert the relationships dataframe to a list of dicts for yfiles-jupyter-graphs."""
    relationships = []
    for _, row in df.iterrows():
        # Create a dictionary for each row
        relationships.append({
            "start": row["source"],
            "end": row["target"],
            "properties": row.to_dict(),
        })
    return relationships


w = GraphWidget()
w.directed = True
w.nodes = convert_entities_to_dicts(entity_df)
w.edges = convert_relationships_to_dicts(relationship_df)


Note: you may need to restart the kernel to use updated packages.


In [8]:
# show title on the node
w.node_label_mapping = "title"


# map community to a color
def community_to_color(community):
    """Map a community to a color."""
    colors = [
        "crimson",
        "darkorange",
        "indigo",
        "cornflowerblue",
        "cyan",
        "teal",
        "green",
    ]
    return (
        colors[int(community) % len(colors)] if community is not None else "lightgray"
    )


def edge_to_source_community(edge):
    """Get the community of the source node of an edge."""
    source_node = next(
        (entry for entry in w.nodes if entry["properties"]["title"] == edge["start"]),
        None,
    )
    source_node_community = source_node["properties"]["community"]
    return source_node_community if source_node_community is not None else None


w.node_color_mapping = lambda node: community_to_color(node["properties"]["community"])
w.edge_color_mapping = lambda edge: community_to_color(edge_to_source_community(edge))
# map size data to a reasonable factor
w.node_scale_factor_mapping = lambda node: 0.5 + node["properties"]["size"] * 1.5 / 20
# use weight for edge thickness
w.edge_thickness_factor_mapping = "weight"


In [9]:
# Use the circular layout for this visualization. For larger graphs, the default organic layout is often preferrable.
w.circular_layout()


In [11]:
entity_node_df =read_parquet_from_azure(FINAL_NODE_PATH)

entity_node_df.head()


✅ Successfully read: output/create_final_nodes.parquet


,id,human_readable_id,title,community,level,degree,x,y
0,df671504-b4f4-4902-9b63-2de11870e1b1,0,MICROSOFT,2,0,2,0.0,0.0
1,df671504-b4f4-4902-9b63-2de11870e1b1,0,MICROSOFT,11,1,2,0.0,0.0
2,99cfec08-f072-47e8-a1ec-5de2e4204a87,1,DRAM,2,0,9,0.0,0.0
3,99cfec08-f072-47e8-a1ec-5de2e4204a87,1,DRAM,8,1,9,0.0,0.0
4,d7975622-0be1-472a-b013-56c7d5a93890,2,NAND,2,0,4,0.0,0.0


In [ ]:


# setup (see also ../../local_search.ipynb)
entities = read_indexer_entities(entity_node_df, community_df,COMMUNITY_LEVEL)

from graphrag.vector_stores.azure_ai_search import AzureAISearchVectorStore

# description_embedding_store = LanceDBVectorStore(
#     collection_name="default-entity-description",
# )
# description_embedding_store.connect(db_uri=LANCEDB_URI)

# to connect to a remote db, specify url and port values.
description_embedding_store = AzureAISearchVectorStore(collection_name="default-community-full_content",)
description_embedding_store.connect(url = "",api_key ="" )


covariate_df = read_parquet_from_azure(COVARIATE_TABLE)
claims = read_indexer_covariates(covariate_df)
covariates = {"claims": claims}

report_df = read_parquet_from_azure(COMMUNITY_REPORT_TABLE)
reports = read_indexer_reports(report_df, community_df, COMMUNITY_LEVEL)

text_unit_df = read_parquet_from_azure(TEXT_UNIT_TABLE)
text_units = read_indexer_text_units(text_unit_df)

import json 
# Load configuration from config.json
with open("C:/Users/dipankar.nath/Downloads/Graphrag/pro_code/config.json") as config_file:
    config = json.load(config_file)

api_key =  config["llm"]["api_key"]
llm_model =  config["llm"]["engine"]
azure_endpoint = config["llm"]["azure_endpoint"]
api_version= config["llm"]["api_version"]

embedding_model =  config["embed"]["model"]
embed_endpoint = config["embed"]["azure_endpoint"]
embed_version= config["embed"]["api_version"]


llm = ChatOpenAI(
    api_key=api_key,
    api_base =azure_endpoint,
    api_version=api_version,
    model=llm_model,
    api_type=OpenaiApiType.AzureOpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=embed_endpoint,
    api_type=OpenaiApiType.AzureOpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    api_version=embed_version,
    max_retries=20,
)

context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

llm_params = {
    "max_tokens": 2_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.0,
}

search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraphs",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)


✅ Successfully read: output/create_final_covariates.parquet
✅ Successfully read: output/create_final_community_reports.parquet
✅ Successfully read: output/create_final_text_units.parquet


In [ ]:

pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

report_df["full_content_json"].head()


0    {\n    "title": "DRAM Market Dynamics and Rela...
1    {\n    "title": "China Mobile OEMs and Micron:...
2    {\n    "title": "Q423 and 64GB RD4 Market Anal...
3    {\n    "title": "Microsoft and De Dios & Assoc...
4    {\n    "title": "HBMX and TSV Technology Compe...
Name: full_content_json, dtype: object

In [13]:
result = await search_engine.asearch("When is the supplier inventory for DRAMs expected to be the highest?")
print(result.response)


The supplier inventory for DRAMs is expected to be the highest in Q3 2023. This projection is based on the anticipated increase in production and a temporary slowdown in demand, which will likely lead to higher inventory levels during this period. The buildup in inventory is a typical cyclical response observed in the semiconductor industry, where suppliers adjust their production rates based on anticipated demand changes.

### Factors Influencing DRAM Inventory Levels

1. **Production Adjustments**: Suppliers often increase production in anticipation of high demand periods such as new technology rollouts or seasonal sales increases. However, if the demand does not meet expectations, this results in higher inventory levels.

2. **Market Demand Fluctuations**: The demand for DRAMs can fluctuate based on various factors including economic conditions, consumer spending habits, and technological advancements. A lower than expected demand leads to higher inventory levels.

3. **Technologica

In [14]:
question = "When is the supplier inventory for DRAMs expected to be the highest?"
result = await search_engine.asearch(question)
print(result.response)


The supplier inventory for DRAMs is expected to be the highest in Q3 2023. This projection is based on the anticipated increase in production and a temporary slowdown in demand, which will likely lead to higher inventory levels during this period. The buildup in inventory is a typical seasonal pattern observed in the DRAM market, where manufacturers adjust their production schedules in response to anticipated changes in market demand.

### Factors Influencing DRAM Inventory Levels

1. **Production Adjustments**: Manufacturers often increase production in anticipation of high demand periods such as the back-to-school season or during major technology upgrades in industries.
2. **Market Demand Fluctuations**: Demand for DRAMs can vary significantly due to factors such as new mobile device launches, server upgrade cycles in the IT industry, and general economic conditions.
3. **Technological Advancements**: As new memory technologies and higher capacity DRAMs are developed, inventory leve

In [15]:
print(result)


SearchResult(response="The supplier inventory for DRAMs is expected to be the highest in Q3 2023. This projection is based on the anticipated increase in production and a temporary slowdown in demand, which will likely lead to higher inventory levels during this period. The buildup in inventory is a typical seasonal pattern observed in the DRAM market, where manufacturers adjust their production schedules in response to anticipated changes in market demand.\n\n### Factors Influencing DRAM Inventory Levels\n\n1. **Production Adjustments**: Manufacturers often increase production in anticipation of high demand periods such as the back-to-school season or during major technology upgrades in industries.\n2. **Market Demand Fluctuations**: Demand for DRAMs can vary significantly due to factors such as new mobile device launches, server upgrade cycles in the IT industry, and general economic conditions.\n3. **Technological Advancements**: As new memory technologies and higher capacity DRAMs 

In [16]:
"""
Helper function to visualize the result context with `yfiles-jupyter-graphs`.

The dataframes are converted into supported nodes and relationships lists and then passed to yfiles-jupyter-graphs.
Additionally, some values are mapped to visualization properties.
"""


def show_graph(result):
    """Visualize the result context with yfiles-jupyter-graphs."""
    from yfiles_jupyter_graphs import GraphWidget

    if (
        "entities" not in result.context_data
        or "relationships" not in result.context_data
    ):
        msg = "The passed results do not contain 'entities' or 'relationships'"
        raise ValueError(msg)

    # converts the entities dataframe to a list of dicts for yfiles-jupyter-graphs
    def convert_entities_to_dicts(df):
        """Convert the entities dataframe to a list of dicts for yfiles-jupyter-graphs."""
        nodes_dict = {}
        for _, row in df.iterrows():
            # Create a dictionary for each row and collect unique nodes
            node_id = row["entity"]
            if node_id not in nodes_dict:
                nodes_dict[node_id] = {
                    "id": node_id,
                    "properties": row.to_dict(),
                }
        return list(nodes_dict.values())

    # converts the relationships dataframe to a list of dicts for yfiles-jupyter-graphs
    def convert_relationships_to_dicts(df):
        """Convert the relationships dataframe to a list of dicts for yfiles-jupyter-graphs."""
        relationships = []
        for _, row in df.iterrows():
            # Create a dictionary for each row
            relationships.append({
                "start": row["source"],
                "end": row["target"],
                "properties": row.to_dict(),
            })
        return relationships

    w = GraphWidget()
    # use the converted data to visualize the graph
    w.nodes = convert_entities_to_dicts(result.context_data["entities"])
    w.edges = convert_relationships_to_dicts(result.context_data["relationships"])
    w.directed = True
    # show title on the node
    w.node_label_mapping = "entity"
    # use weight for edge thickness
    w.edge_thickness_factor_mapping = "weight"
    display(w)


show_graph(result)


ValueError: The passed results do not contain 'entities' or 'relationships'